In [0]:
# Databricks notebook source

# COMMAND ----------
# Formula 1 Racing Data Platform
# Notebook: silver\05_seasons
#
# Purpose:
# Process seasons table from Bronze layer to Silver layer.

In [0]:
# COMMAND ----------
# Define table name and primary key

TABLE_NAME = "seasons"
PRIMARY_KEY = "year"

In [0]:
%run ../common/00_configure_catalog 

In [0]:
%run ../common/01_logger

In [0]:
%run ./00_utils

In [0]:
# COMMAND ----------
# Import

import time

In [0]:
# COMMAND ----------
# Store start time for the logger

start_time = time.time()

In [0]:
# COMMAND ----------
# Read Bronze table

bronze_df = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.{TABLE_NAME}"
)

print(f"Rows read: {bronze_df.count()}")

In [0]:
# COMMAND ----------
# Initial validation

print("=" * 70)
print("🔎 Initial Validation")
print("=" * 70)

print(f"📄 Rows: {bronze_df.count()}")

print(
    f"🔑 Primary Key Valid: "
    f"{validate_primary_key(bronze_df, PRIMARY_KEY)}"
)

print(
    f"👥 Duplicate Rows: "
    f"{bronze_df.count() - bronze_df.dropDuplicates().count()}"
)

In [0]:
# COMMAND ----------
# Data cleaning

silver_df = (
    bronze_df
    .transform(remove_duplicates)
    .transform(trim_string_columns)
)

In [0]:
# COMMAND ----------
# Business rules

from pyspark.sql.functions import col

silver_df = silver_df.filter(
    col(PRIMARY_KEY).isNotNull()
)

In [0]:
# COMMAND ----------
# Write silver

write_delta(
    silver_df,
    f"{CATALOG}.{SILVER_SCHEMA}.{TABLE_NAME}"
)

In [0]:
# COMMAND ----------
# Post validation

silver_table = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.{TABLE_NAME}"
)

print("=" * 70)
print("Post Validation")
print("=" * 70)

print(f"Rows written: {silver_table.count()}")

print(
    f"Primary Key Valid: "
    f"{validate_primary_key(silver_table, PRIMARY_KEY)}"
)

In [0]:
# COMMAND ----------
# Log pipeline execution

execution_time_ms = int(
    (time.time() - start_time) * 1000
)

log_pipeline_execution(
    layer="silver",
    table_name=TABLE_NAME,
    rows_read=bronze_df.count(),
    rows_written=silver_df.count(),
    execution_time_ms=execution_time_ms
)